In [3]:
#pip install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece pandas scikit-learn

In [4]:
#pip install pandas==2.2.3

In [5]:
#pip install torch torchvision torchaudio

In [6]:
import warnings
warnings.filterwarnings("ignore")

In [7]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer, SFTConfig

In [8]:
#AutoTokenizer converts normal text into tokens that the model understands.
#AutoModelForCausalLM loads a causal language model.Causal language models predict the next token.
#BitsAndBytesConfig is used for quantization.
#FP32 → 32 bits
#FP16 → 16 bits
#BF16 → 16 bits
#LoraConfig tells PEFT how you want to apply LoRA.
#lora_alpha controls the strength/scaling of LoRA updates.
#lora_dropout helps reduce overfitting.
#task_type="CAUSAL_LM" tells PEFT that we are fine-tuning a text-generation model.

In [9]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.14.0+cpu
CUDA available: False


In [10]:
df = pd.read_csv(
    "https://github.com/laxmimerit/All-CSV-ML-Data-Files-Download/raw/master/amazon_product_details.csv",
    usecols=["category", "about_product", "product_name"]
)

In [11]:
df

,product_name,category,about_product
0,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,High Compatibility : Compatible With iPhone 12...
1,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,"Compatible with all Type C enabled devices, be..."
2,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,【 Fast Charger& Data Sync】-With built-in safet...
3,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,The boAt Deuce USB 300 2 in 1 cable is compati...
4,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,[CHARGE & SYNC FUNCTION]- This cable comes wit...
...,...,...,...
1460,Noir Aqua - 5pcs PP Spun Filter + 1 Spanner | ...,Home&Kitchen|Kitchen&HomeAppliances|WaterPurif...,SUPREME QUALITY 90 GRAM 3 LAYER THIK PP SPUN F...
1461,Prestige Delight PRWO Electric Rice Cooker (1 ...,Home&Kitchen|Kitchen&HomeAppliances|SmallKitch...,"230 Volts, 400 watts, 1 Year"
1462,Bajaj Majesty RX10 2000 Watts Heat Convector R...,"Home&Kitchen|Heating,Cooling&AirQuality|RoomHe...",International design and styling|Two heat sett...
1463,Havells Ventil Air DSP 230mm Exhaust Fan (Pist...,"Home&Kitchen|Heating,Cooling&AirQuality|Fans|E...",Fan sweep area: 230 MM ; Noise level: (40 - 45...


In [12]:
df['category']

0       Computers&Accessories|Accessories&Peripherals|...
1       Computers&Accessories|Accessories&Peripherals|...
2       Computers&Accessories|Accessories&Peripherals|...
3       Computers&Accessories|Accessories&Peripherals|...
4       Computers&Accessories|Accessories&Peripherals|...
                              ...                        
1460    Home&Kitchen|Kitchen&HomeAppliances|WaterPurif...
1461    Home&Kitchen|Kitchen&HomeAppliances|SmallKitch...
1462    Home&Kitchen|Heating,Cooling&AirQuality|RoomHe...
1463    Home&Kitchen|Heating,Cooling&AirQuality|Fans|E...
1464    Home&Kitchen|Kitchen&HomeAppliances|SmallKitch...
Name: category, Length: 1465, dtype: str

In [13]:
print("Product Name:")
print(df.loc[0, "product_name"])

print("\nCategory:")
print(df.loc[0, "category"])

print("\nDescription:")
print(df.loc[0, "about_product"])

Product Name:
Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)

Category:
Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables

Description:
High Compatibility : Compatible With iPhone 12, 11, X/XsMax/Xr ,iPhone 8/8 Plus,iPhone 7/7 Plus,iPhone 6s/6s Plus,iPhone 6/6 Plus,iPhone 5/5s/5c/se,iPad Pro,iPad Air 1/2,iPad mini 1/2/3,iPod nano7,iPod touch and more apple devices.|Fast Charge&Data Sync : It can charge and sync simultaneously at a rapid speed, Compatible with any charging adaptor, multi-port charging station or power bank.|Durability : Durable nylon braided design with premium aluminum housing and toughened nylon fiber wound tightly around the cord lending it superior durability and adding a bit to its flexibility.|High Security Level : It is designed to fully protect your device from damaging excessive current.Copper core thick+Multilayer 

In [14]:
df['category'] = df['category'].apply(lambda x: x.split('|')[-1])


In [15]:
df

,product_name,category,about_product
0,Wayona Nylon Braided USB to Lightning Fast Cha...,USBCables,High Compatibility : Compatible With iPhone 12...
1,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,USBCables,"Compatible with all Type C enabled devices, be..."
2,Sounce Fast Phone Charging Cable & Data Sync U...,USBCables,【 Fast Charger& Data Sync】-With built-in safet...
3,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,USBCables,The boAt Deuce USB 300 2 in 1 cable is compati...
4,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,USBCables,[CHARGE & SYNC FUNCTION]- This cable comes wit...
...,...,...,...
1460,Noir Aqua - 5pcs PP Spun Filter + 1 Spanner | ...,WaterPurifierAccessories,SUPREME QUALITY 90 GRAM 3 LAYER THIK PP SPUN F...
1461,Prestige Delight PRWO Electric Rice Cooker (1 ...,Rice&PastaCookers,"230 Volts, 400 watts, 1 Year"
1462,Bajaj Majesty RX10 2000 Watts Heat Convector R...,HeatConvectors,International design and styling|Two heat sett...
1463,Havells Ventil Air DSP 230mm Exhaust Fan (Pist...,ExhaustFans,Fan sweep area: 230 MM ; Noise level: (40 - 45...


In [16]:
def create_prompt(row):

    prompt = f"""### Instruction:
Generate a clear and informative product description for the following Amazon product.

### Product Name:
{row['product_name']}

### Category:
{row['category']}

### Response:
{row['about_product']}"""

    return prompt

In [17]:
df["text"] = df.apply(create_prompt, axis=1)

In [18]:
df

,product_name,category,about_product,text
0,Wayona Nylon Braided USB to Lightning Fast Cha...,USBCables,High Compatibility : Compatible With iPhone 12...,### Instruction:\nGenerate a clear and informa...
1,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,USBCables,"Compatible with all Type C enabled devices, be...",### Instruction:\nGenerate a clear and informa...
2,Sounce Fast Phone Charging Cable & Data Sync U...,USBCables,【 Fast Charger& Data Sync】-With built-in safet...,### Instruction:\nGenerate a clear and informa...
3,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,USBCables,The boAt Deuce USB 300 2 in 1 cable is compati...,### Instruction:\nGenerate a clear and informa...
4,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,USBCables,[CHARGE & SYNC FUNCTION]- This cable comes wit...,### Instruction:\nGenerate a clear and informa...
...,...,...,...,...
1460,Noir Aqua - 5pcs PP Spun Filter + 1 Spanner | ...,WaterPurifierAccessories,SUPREME QUALITY 90 GRAM 3 LAYER THIK PP SPUN F...,### Instruction:\nGenerate a clear and informa...
1461,Prestige Delight PRWO Electric Rice Cooker (1 ...,Rice&PastaCookers,"230 Volts, 400 watts, 1 Year",### Instruction:\nGenerate a clear and informa...
1462,Bajaj Majesty RX10 2000 Watts Heat Convector R...,HeatConvectors,International design and styling|Two heat sett...,### Instruction:\nGenerate a clear and informa...
1463,Havells Ventil Air DSP 230mm Exhaust Fan (Pist...,ExhaustFans,Fan sweep area: 230 MM ; Noise level: (40 - 45...,### Instruction:\nGenerate a clear and informa...


In [19]:
dataset_df = df[["text"]]

In [20]:
dataset_df

,text
0,### Instruction:\nGenerate a clear and informa...
1,### Instruction:\nGenerate a clear and informa...
2,### Instruction:\nGenerate a clear and informa...
3,### Instruction:\nGenerate a clear and informa...
4,### Instruction:\nGenerate a clear and informa...
...,...
1460,### Instruction:\nGenerate a clear and informa...
1461,### Instruction:\nGenerate a clear and informa...
1462,### Instruction:\nGenerate a clear and informa...
1463,### Instruction:\nGenerate a clear and informa...


In [21]:
train_df, test_df = train_test_split(
    dataset_df,
    test_size=0.1,
    random_state=42
)

In [22]:
print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))

Training samples: 1318
Testing samples: 147


In [23]:
train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)

In [24]:
model_name = "microsoft/phi-2"

In [25]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

In [26]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [27]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)

In [28]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

In [29]:
model.config.use_cache = False

In [30]:
model = prepare_model_for_kbit_training(model)

In [31]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "dense"
    ],

    task_type="CAUSAL_LM"
)

In [32]:
training_args = SFTConfig(
    output_dir="./phi2-amazon-qlora",

    num_train_epochs=1,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    logging_steps=10,

    save_steps=100,

    eval_strategy="steps",

    eval_steps=100,

    save_total_limit=2,

    fp16=True,

    max_length=512,

    dataset_text_field="text",

    report_to="none"
)

In [33]:
small_train_dataset = train_dataset.select(
    range(min(1000, len(train_dataset)))
)

small_test_dataset = test_dataset.select(
    range(min(100, len(test_dataset)))
)

In [34]:
trainer = SFTTrainer(
    model=model,

    args=training_args,

    train_dataset=small_train_dataset,

    eval_dataset=small_test_dataset,

    peft_config=peft_config,

    processing_class=tokenizer
)

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

In [35]:
trainer.model.print_trainable_parameters()

trainable params: 10,485,760 || all params: 2,790,169,600 || trainable%: 0.3758


In [ ]:
trainer.train()

In [ ]:
  trainer.model.save_pretrained(
    "./phi2-amazon-finetuned"
)

tokenizer.save_pretrained(
    "./phi2-amazon-finetuned"
)

In [ ]:
product_name = "Noise Cancelling Wireless Headphones"

category = "Electronics"

In [ ]:
prompt = f"""### Instruction:
Generate a clear and informative product description for the following Amazon product.

### Product Name:
{product_name}

### Category:
{category}

### Response:
"""

In [ ]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

In [ ]:
inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

In [ ]:
model.eval()

with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1
    )

In [ ]:
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)